# LatentSync 1.6 嘴型同步（Colab GPU）

建議主要流程：WSL + `scripts/runner.py`。這份 Notebook 是獨立的手動入口，無須先發布 GitHub 儲存庫。

1. 登入 Google，選 **執行階段 → 變更執行階段類型 → GPU → A100**（L4 亦可嘗試）。1.6 官方最低 18 GB VRAM，T4 不足。
2. 依序執行下方儲存格；首次會下載模型、建立獨立 Python 3.10 環境。
3. 上傳有清楚單人臉部的影片，以及替換的語音音檔。音檔取代原聲。
4. 預覽並下載 MP4 和 report.json，然後 **執行階段 → 中斷連線並刪除執行階段**。

此檔內嵌同版本 Python 原始碼，由 `scripts/build_notebook.py` 產生。修改原始碼後請重新產生 Notebook。
2026-10-24 已以相同 worker 從 WSL CLI 完成 Colab A100 40GB、LatentSync 1.6、FP16 真實推論，成功下載 2.32 秒有聲 MP4並自動停止 session。Notebook已驗證語法與輸出串流，未單獨在網頁介面端到端實跑；請先用短片確認素材品質。
執行會使用 Colab 資源；Colab FAQ 禁止 creating deepfakes，請確認用途符合服務規則。


In [ ]:
#@title 1. 展開執行程式（只寫入此 Notebook 的工具目錄）
from pathlib import Path
import json, subprocess, sys, uuid, shutil
SCRIPT_DIR = Path('/content/latentsync-skill-tools')
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
SOURCES = {'config.py': '"""Shared, dependency-free LatentSync job options for local and Colab execution."""\nfrom __future__ import annotations\n\nimport argparse\nimport math\n\nDEFAULTS = {\n    "model_version": "1.6",\n    "inference_steps": 20,\n    "guidance_scale": 1.5,\n    "seed": 1247,\n    "precision": "auto",\n    "enable_deepcache": False,\n    "video_start": 0.0,\n    "audio_start": 0.0,\n    "duration": None,\n    "length_mode": "shortest",\n    "audio_offset": 0.0,\n    "audio_gain_db": 0.0,\n    "max_side": 1280,\n    "segment_seconds": 0.0,\n    "crf": 18,\n    "preset": "medium",\n    "audio_bitrate": "192k",\n    "vae_slicing": False,\n    "keep_intermediates": False,\n}\n\nPROFILES = {\n    "preview": {"model_version": "1.6", "inference_steps": 10},\n    "balanced": {"model_version": "1.6", "inference_steps": 20},\n    "quality": {"model_version": "1.6", "inference_steps": 30},\n}\n\nCHOICES = {\n    "model_version": ("1.6", "1.5"),\n    "precision": ("auto", "fp16", "fp32"),\n    "length_mode": ("shortest", "audio", "video"),\n    "preset": ("ultrafast", "superfast", "veryfast", "faster", "fast", "medium", "slow", "slower", "veryslow"),\n    "audio_bitrate": ("96k", "128k", "160k", "192k", "256k", "320k"),\n}\n\nHELP = {\n    "model_version": "Matched upstream checkpoint/config version (default 1.6; never auto-downgraded).",\n    "inference_steps": "Diffusion steps 1..100; usual 20..50, preview profile 10.",\n    "guidance_scale": "Audio guidance 1..5; upstream recommends 1..3 (default 1.5).",\n    "seed": "Random seed 0..4294967295; -1 chooses a fresh seed and records it.",\n    "precision": "auto uses fp16 on supported CUDA GPUs; fp32 needs more VRAM.",\n    "enable_deepcache": "Enable upstream DeepCache acceleration; may change output quality.",\n    "video_start": "Trim this many seconds from the source video (>=0).",\n    "audio_start": "Trim this many seconds from the source audio (>=0).",\n    "duration": "Maximum output seconds after alignment; omitted uses available media.",\n    "length_mode": "shortest: trim both; audio: loop video to audio; video: pad/trim audio.",\n    "audio_offset": "Seconds: positive delays audio with silence; negative trims more audio.",\n    "audio_gain_db": "Gain applied to replacement audio, -60..24 dB; 0 preserves level.",\n    "max_side": "Maximum input long edge (default1280); 0 preserves; otherwise even256..4096.",\n    "segment_seconds": "0 processes as one clip; >=1 splits on25fps boundaries (possible seams).",\n    "crf": "Final H.264 CRF 0..51; smaller means larger, cleaner files (default18).",\n    "preset": "Final x264 encoding speed/compression tradeoff, not diffusion quality.",\n    "audio_bitrate": "Final AAC bitrate (default192k). Model conditioning remains16kHz mono.",\n    "vae_slicing": "Enable diffusers VAE slicing; effect on peak memory depends on workload.",\n    "keep_intermediates": "Retain normalized/segment files in the remote job directory.",\n}\n\n\ndef resolve_options(raw: dict | None = None, profile: str = "balanced") -> dict:\n    if profile not in PROFILES:\n        raise ValueError(f"Unknown profile: {profile!r}; choose {\', \'.join(PROFILES)}")\n    if raw is None:\n        raw = {}\n    if not isinstance(raw, dict):\n        raise ValueError("options must be a JSON object")\n    unknown = set(raw) - set(DEFAULTS)\n    if unknown:\n        raise ValueError(f"Unknown options: {\', \'.join(sorted(map(str, unknown)))}")\n    result = {**DEFAULTS, **PROFILES[profile], **raw}\n    for key, choices in CHOICES.items():\n        if not isinstance(result[key], str) or result[key] not in choices:\n            raise ValueError(f"{key} must be one of {\', \'.join(choices)}")\n    for key in ("enable_deepcache", "vae_slicing", "keep_intermediates"):\n        if type(result[key]) is not bool:\n            raise ValueError(f"{key} must be a JSON boolean (true/false)")\n    for key, lower, upper in (("inference_steps", 1, 100), ("seed", -1, 2**32-1), ("crf", 0, 51), ("max_side", 0, 4096)):\n        value = result[key]\n        if type(value) is not int or not lower <= value <= upper:\n            raise ValueError(f"{key} must be an integer in {lower}..{upper}")\n    if result["max_side"] != 0 and (result["max_side"] < 256 or result["max_side"] % 2):\n        raise ValueError("max_side must be 0 or an even integer in 256..4096")\n    for key in ("guidance_scale", "video_start", "audio_start", "duration", "audio_offset", "audio_gain_db", "segment_seconds"):\n        value = result[key]\n        if key == "duration" and value is None:\n            continue\n        if type(value) not in (int, float) or not math.isfinite(value):\n            raise ValueError(f"{key} must be a finite number")\n        result[key] = float(value)\n    if not 1 <= result["guidance_scale"] <= 5:\n        raise ValueError("guidance_scale must be in 1..5; upstream recommends 1..3")\n    if result["video_start"] < 0 or result["audio_start"] < 0:\n        raise ValueError("video_start and audio_start must be >= 0")\n    if result["duration"] is not None and result["duration"] < 0.64:\n        raise ValueError("duration must be >= 0.64 seconds (16 frames at25fps)")\n    if not -60 <= result["audio_gain_db"] <= 24:\n        raise ValueError("audio_gain_db must be in -60..24")\n    if result["segment_seconds"] != 0 and result["segment_seconds"] < 1:\n        raise ValueError("segment_seconds must be 0 (disabled) or >= 1")\n    return result\n\n\ndef add_option_arguments(parser: argparse.ArgumentParser) -> None:\n    group = parser.add_argument_group("model, media preparation and encoding options")\n    for key, default in DEFAULTS.items():\n        kwargs = {"dest": key, "default": None, "help": HELP[key]}\n        if type(default) is bool:\n            kwargs["action"] = argparse.BooleanOptionalAction\n        elif key in CHOICES:\n            kwargs["choices"] = CHOICES[key]\n        else:\n            kwargs["type"] = float if default is None else type(default)\n        group.add_argument("--" + key.replace("_", "-"), **kwargs)\n\n\ndef options_from_namespace(args: argparse.Namespace) -> dict:\n    return {key: getattr(args, key) for key in DEFAULTS if getattr(args, key, None) is not None}\n', 'remote_runtime.py': '#!/usr/bin/env python3\n"""Colab-only bootstrap and LatentSync worker; media helpers also run without ML.\n\nThe adapter deliberately specifies dtype instead of copying upstream\'s T4->fp32\nheuristic. Upstream files are never patched. Every upstream invocation has an\nisolated ASCII working directory because upstream uses unquoted shell commands\nand recursively replaces its own ``temp`` directory.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport traceback\nimport urllib.request\nimport uuid\nimport zipfile\n\nfrom config import resolve_options\n\nUPSTREAM_URL = "https://github.com/bytedance/LatentSync.git"\nUPSTREAM_SHA = "a229c3948406bc2cf6eaf4873e662e70c6a04746"\nVAE_REPO = "stabilityai/sd-vae-ft-mse"\nVAE_REVISION = "31f26fdeee1355a5c34592e401dd41e45d25a493"\nMODELS = {\n    "1.5": {"repo": "ByteDance/LatentSync-1.5", "revision": "32a20d29aead0498e3e885e90dbbe8027da1b61b",\n            "config": "configs/unet/stage2.yaml", "resolution": 256, "minimum_gib": 8},\n    "1.6": {"repo": "ByteDance/LatentSync-1.6", "revision": "c42c7e6c8e9c213626389fa7d9a3c444b8536353",\n            "config": "configs/unet/stage2_512.yaml", "resolution": 512, "minimum_gib": 18},\n}\nBUFFALO_URL = "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip"\nFACE_FILES = ("det_10g.onnx", "2d106det.onnx")\n# All environment/bootstrap pins live here. Increment INSTALLER_REVISION if the\n# installation algorithm changes without changing a version below.\nINSTALLER_REVISION = 2\nPYTHON_VERSION = "3.10"\nUV_REQUIREMENT = "uv==0.8.22"\nBUILD_REQUIREMENTS = [\n    "pip==25.0.1", "setuptools==75.8.0", "wheel==0.45.1",\n    "Cython==3.0.12", "numpy==1.26.4",\n]\nTORCH_REQUIREMENTS = ["torch==2.5.1", "torchvision==0.20.1"]\nTORCH_INDEX_URL = "https://download.pytorch.org/whl/cu121"\nINSIGHTFACE_REQUIREMENTS = ["insightface==0.7.3"]\nINSIGHTFACE_SUPPORT_REQUIREMENTS = [\n    "tqdm==4.67.1", "requests==2.32.3", "easydict==1.13", "prettytable==3.12.0",\n]\n# Inference-only requirements. Training/evaluation/Gradio are intentionally not\n# installed. Pin OpenCV/albumentations to avoid silently reintroducing NumPy 2.\nREQUIREMENTS = [\n    "diffusers==0.32.2", "transformers==4.48.0", "decord==0.6.0",\n    "accelerate==0.26.1", "einops==0.7.0", "omegaconf==2.3.0",\n    "opencv-python-headless==4.9.0.80", "ffmpeg-python==0.2.0",\n    "imageio==2.31.1", "imageio-ffmpeg==0.5.1", "huggingface-hub==0.30.2",\n    "kornia==0.8.0", "onnxruntime-gpu==1.21.0",\n    "onnx==1.17.0", "albumentations==1.4.3", "scipy==1.12.0",\n    "scikit-image==0.22.0", "scikit-learn==1.4.2", "matplotlib==3.8.4",\n    "soundfile==0.12.1", "more-itertools==10.2.0", "DeepCache==0.1.1",\n    "protobuf==4.25.6", "packaging==24.2", "safetensors==0.5.2",\n    "tokenizers==0.21.0", "python-dateutil==2.9.0.post0", "Pillow==10.4.0",\n]\n\n\ndef environment_manifest():\n    """Complete direct installation contract, including bootstrap and indexes."""\n    return {\n        "installer_revision": INSTALLER_REVISION,\n        "python": PYTHON_VERSION,\n        "uv": UV_REQUIREMENT,\n        "build": BUILD_REQUIREMENTS,\n        "torch": TORCH_REQUIREMENTS,\n        "torch_index": TORCH_INDEX_URL,\n        "inference": REQUIREMENTS,\n        "insightface": INSIGHTFACE_REQUIREMENTS,\n        "insightface_support": INSIGHTFACE_SUPPORT_REQUIREMENTS,\n    }\n\n\ndef environment_fingerprint():\n    encoded = json.dumps(environment_manifest(), sort_keys=True, separators=(",", ":")).encode()\n    return hashlib.sha256(encoded).hexdigest()[:12]\n\n\nENV_VERSION = environment_fingerprint()\n\n\ndef run_cmd(args, *, cwd=None, env=None, capture=False):\n    """Never pass user input through a shell; propagate every process failure."""\n    args = [str(arg) for arg in args]\n    if not capture:\n        print("+ " + " ".join(args[:6]) + (" ..." if len(args) > 6 else ""), flush=True)\n    result = subprocess.run(args, cwd=cwd, env=env, check=True, text=True,\n                            encoding="utf-8", errors="replace", capture_output=capture)\n    return result.stdout if capture else result\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef probe(path):\n    """Return stream metadata; never use container duration when stream is known."""\n    data = json.loads(run_cmd(["ffprobe", "-v", "error", "-show_streams", "-show_format",\n                               "-of", "json", path], capture=True))\n    streams = data.get("streams", [])\n    result = {"path": str(path), "streams": streams}\n    container_duration = float(data.get("format", {}).get("duration", 0) or 0)\n    for kind in ("video", "audio"):\n        stream = next((s for s in streams if s.get("codec_type") == kind), None)\n        if stream:\n            result[kind] = stream\n            result[kind + "_duration"] = float(stream.get("duration", container_duration) or 0)\n    return result\n\n\ndef select_duration(video_duration, audio_duration, options):\n    video_left = video_duration - options["video_start"]\n    raw_audio_left = audio_duration - options["audio_start"]\n    audio_left = raw_audio_left + options["audio_offset"]\n    if video_left <= 0 or raw_audio_left <= 0 or audio_left <= 0:\n        raise ValueError("Start time / negative audio offset removes the entire video or audio.")\n    mode = options["length_mode"]\n    duration = {"shortest": min(video_left, audio_left), "video": video_left,\n                "audio": audio_left}[mode]\n    if options["duration"] is not None:\n        duration = min(duration, options["duration"])\n    # All cut positions use exact 25 FPS frame boundaries. Do not generate a\n    # partial frame or let rounding extend the requested source duration.\n    frames = int(math.floor(duration * 25 + 1e-7))\n    if frames < 16:\n        raise ValueError("The selected duration must contain at least 0.64 seconds (16 frames at 25 FPS).")\n    return frames / 25\n\n\ndef segment_ranges(duration, segment_seconds):\n    total = int(round(duration * 25))\n    chunk = max(1, int(math.floor(segment_seconds * 25))) if segment_seconds else total\n    ranges = [(start / 25, min(chunk, total - start) / 25) for start in range(0, total, chunk)]\n    if len(ranges) > 1 and ranges[-1][1] < 0.64:\n        tail = ranges.pop()\n        previous = ranges.pop()\n        ranges.append((previous[0], round(previous[1] + tail[1], 8)))\n    return ranges\n\n\ndef prepare_media(video, audio, workspace, options):\n    """Trim, normalize and align media without ever modifying source files."""\n    workspace = Path(workspace)\n    workspace.mkdir(parents=True, exist_ok=True)\n    vmeta, ameta = probe(video), probe(audio)\n    if "video" not in vmeta or "audio" not in ameta:\n        raise ValueError("Video input needs a video stream; audio input needs an audio stream.")\n    duration = select_duration(vmeta["video_duration"], ameta["audio_duration"], options)\n    # Stage a trimmed video before looping, so video_start applies to every loop.\n    filters = ["fps=25"]\n    if options["max_side"]:\n        size = options["max_side"]\n        filters.append(f"scale=w=\'trunc(iw*min(1,{size}/max(iw,ih))/2)*2\':h=\'trunc(ih*min(1,{size}/max(iw,ih))/2)*2\'")\n    else:\n        filters.append("scale=trunc(iw/2)*2:trunc(ih/2)*2")\n    base = workspace / "base.mp4"\n    run_cmd(["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y",\n             "-ss", options["video_start"], "-i", video, "-t",\n             min(duration, vmeta["video_duration"] - options["video_start"]),\n             "-map", "0:v:0", "-an", "-vf", ",".join(filters), "-c:v", "libx264",\n             "-crf", "13", "-preset", "fast", "-pix_fmt", "yuv420p", base])\n    video_out = workspace / "input.mp4"\n    # Explicit forward looping differs from upstream\'s hidden ping-pong loop.\n    run_cmd(["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y",\n             "-stream_loop", "-1", "-i", base, "-frames:v", round(duration * 25), "-map", "0:v:0",\n             "-an", "-c:v", "libx264", "-crf", "13", "-preset", "fast", video_out])\n    afilters = []\n    if options["audio_offset"] < 0:\n        afilters += [f"atrim=start={-options[\'audio_offset\']}", "asetpts=PTS-STARTPTS"]\n    elif options["audio_offset"] > 0:\n        # Sample units preserve sub-millisecond offsets at the target rate.\n        afilters += ["aresample=16000", f"adelay={round(options[\'audio_offset\'] * 16000)}S:all=1"]\n    afilters += [f"volume={options[\'audio_gain_db\']}dB", "apad", f"atrim=duration={duration}"]\n    audio_out = workspace / "audio.wav"\n    run_cmd(["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y",\n             "-ss", options["audio_start"], "-i", audio, "-map", "0:a:0", "-vn",\n             "-af", ",".join(afilters), "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", audio_out])\n    return {"duration": duration, "video": str(video_out), "audio": str(audio_out),\n            "source_video": vmeta, "source_audio": ameta,\n            "segments": segment_ranges(duration, options["segment_seconds"])}\n\n\ndef validate_output(path, expected_duration=None):\n    path = Path(path)\n    if not path.is_file() or path.stat().st_size < 256:\n        raise RuntimeError("Inference did not produce a nonempty video.")\n    metadata = probe(path)\n    if "video" not in metadata or "audio" not in metadata:\n        raise RuntimeError("Output must contain both video and audio streams.")\n    if expected_duration is not None:\n        for kind in ("video", "audio"):\n            if abs(metadata[kind + "_duration"] - expected_duration) > 0.16:\n                raise RuntimeError(f"Output {kind} duration differs from expected duration: {metadata[kind + \'_duration\']} vs {expected_duration}")\n    run_cmd(["ffmpeg", "-v", "error", "-xerror", "-nostdin", "-i", path,\n             "-map", "0:v:0", "-map", "0:a:0", "-f", "null", "-"])\n    return metadata\n\n\ndef managed_root(root):\n    root = Path(root).expanduser().resolve()\n    if sys.platform != "linux" or not Path("/content").is_dir():\n        raise RuntimeError("Setup/inference runs inside a Linux Colab runtime, not on your local computer.")\n    # Keep mutable environment/locks away from Drive, and enforce safe shell paths.\n    if not str(root).startswith("/content/") or not all(c.isascii() and (c.isalnum() or c in "/_-.") for c in str(root)):\n        raise ValueError("--root must be an ASCII path below /content (letters/digits/_/-/. only).")\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / ".latentsync-skill-managed"\n    if not marker.exists() and any(root.iterdir()):\n        raise ValueError("Refusing to adopt a nonempty directory without the LatentSync managed marker.")\n    marker.touch(exist_ok=True)\n    return root\n\n\n@contextlib.contextmanager\ndef root_lock(root):\n    import fcntl\n    with (root / ".runtime.lock").open("a") as handle:\n        try:\n            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError as exc:\n            raise RuntimeError("Another setup/job is using this runtime root. Wait for it to finish.") from exc\n        try:\n            yield\n        finally:\n            fcntl.flock(handle, fcntl.LOCK_UN)\n\n\ndef runtime_python(root):\n    return root / ("venv-" + ENV_VERSION) / "bin" / "python"\n\n\ndef child_env(root):\n    env = os.environ.copy()\n    env["HF_HOME"] = str(root / "hf-cache")\n    env["PYTHONNOUSERSITE"] = "1"\n    env["PYTHONUNBUFFERED"] = "1"\n    env["UV_PYTHON_INSTALL_DIR"] = str(root / "python")\n    env["UV_CACHE_DIR"] = str(root / "uv-cache")\n    env.pop("PYTHONPATH", None)\n    return env\n\n\ndef check_dependencies(python, root, env):\n    """Check requirements and actually decode H.264 before marking ready.\n\n    Decord 0.6.0\'s Linux wheel has a stale internal cp36 tag even though its\n    filename advertises py3-none. pip >=24.2 reports this metadata defect.\n    Only that exact warning is eligible for the functional decode check;\n    missing/conflicting requirements and decode failures remain fatal.\n    See https://github.com/dmlc/decord/issues/313.\n    """\n    command = [str(python), "-m", "pip", "check"]\n    checked = subprocess.run(command, env=env, text=True, encoding="utf-8",\n                             errors="replace", stdout=subprocess.PIPE, stderr=subprocess.STDOUT)\n    diagnostics = checked.stdout or ""\n    print(diagnostics, end="" if diagnostics.endswith("\\n") else "\\n", flush=True)\n    known = "decord 0.6.0 is not supported on this platform"\n    lines = [line.strip() for line in diagnostics.splitlines() if line.strip()]\n    if checked.returncode and not (checked.returncode == 1 and lines == [known]):\n        raise subprocess.CalledProcessError(checked.returncode, command, output=diagnostics)\n    with tempfile.TemporaryDirectory(prefix="decode-check-", dir=root) as temporary:\n        video = Path(temporary) / "probe.mp4"\n        run_cmd(["ffmpeg", "-v", "error", "-nostdin", "-y", "-f", "lavfi", "-i",\n                 "testsrc2=size=64x64:rate=25:duration=0.64", "-c:v", "libx264",\n                 "-pix_fmt", "yuv420p", video], env=env)\n        smoke = (\n            "import platform, sys, decord, importlib.metadata\\n"\n            "assert sys.platform == \'linux\' and platform.machine() == \'x86_64\'\\n"\n            "assert decord.__version__ == \'0.6.0\'\\n"\n            + ("assert \'Tag: cp36-cp36m-manylinux2010_x86_64\' in importlib.metadata.distribution(\'decord\').read_text(\'WHEEL\').splitlines()\\n" if checked.returncode else "") +\n            "reader = decord.VideoReader(sys.argv[1], ctx=decord.cpu(0))\\n"\n            "assert len(reader) == 16\\n"\n            "frames = reader.get_batch([0, 15]).asnumpy()\\n"\n            "assert frames.shape == (2, 64, 64, 3) and frames.max() > 0\\n"\n            "print(\'LATENTSYNC_DECORD_DECODE_OK\', flush=True)\\n"\n        )\n        run_cmd([python, "-c", smoke, video], env=env)\n    if checked.returncode:\n        print("Decord 0.6.0 wheel metadata warning accepted after successful Linux H.264 decode.", flush=True)\n\n\ndef setup(root, model_version):\n    root = managed_root(root)\n    with root_lock(root):\n        env = child_env(root)\n        if not shutil.which("ffmpeg") or not shutil.which("git") or not shutil.which("g++"):\n            run_cmd(["apt-get", "update", "-qq"])\n            run_cmd(["apt-get", "install", "-y", "ffmpeg", "git", "build-essential", "libgl1", "libglib2.0-0"])\n        python = runtime_python(root)\n        ready = python.parent.parent / ".dependencies-ready"\n        if not ready.exists():\n            # A distinct bootstrap directory avoids importing a stale uv after\n            # changing its pin. --upgrade also repairs interrupted installations.\n            bootstrap = root / ("bootstrap-" + ENV_VERSION)\n            run_cmd([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",\n                     "--upgrade", "--target", bootstrap, UV_REQUIREMENT], env=env)\n            uv = [sys.executable, "-m", "uv"]\n            uv_env = dict(env, PYTHONPATH=str(bootstrap))\n            if not python.exists():\n                run_cmd(uv + ["venv", "--python", PYTHON_VERSION, "--python-preference", "only-managed",\n                              "--seed", python.parent.parent], env=uv_env)\n            pip = [python, "-m", "pip", "install", "--disable-pip-version-check"]\n            run_cmd(pip + BUILD_REQUIREMENTS, env=env)\n            run_cmd(pip + TORCH_REQUIREMENTS + ["--index-url", TORCH_INDEX_URL], env=env)\n            # Include build pins again so the resolver cannot upgrade NumPy or\n            # setuptools while satisfying a later inference dependency.\n            run_cmd(pip + REQUIREMENTS + BUILD_REQUIREMENTS, env=env)\n            run_cmd(pip + ["--no-build-isolation", "--no-deps"] + INSIGHTFACE_REQUIREMENTS, env=env)\n            # insightface requirements not already provided by the inference set.\n            run_cmd(pip + INSIGHTFACE_SUPPORT_REQUIREMENTS, env=env)\n            check_dependencies(python, root, env)\n            write_json(python.parent.parent / "environment.manifest.json", environment_manifest())\n            ready.write_text(ENV_VERSION, encoding="ascii")\n        repo = root / "upstream"\n        if not (repo / ".git").is_dir():\n            if repo.exists():\n                raise RuntimeError("Incomplete upstream directory exists. Use a fresh --root to recover.")\n            run_cmd(["git", "clone", "--no-checkout", "--filter=blob:none", UPSTREAM_URL, repo])\n            run_cmd(["git", "-C", repo, "checkout", "--detach", UPSTREAM_SHA])\n        revision = run_cmd(["git", "-C", repo, "rev-parse", "HEAD"], capture=True).strip()\n        if revision != UPSTREAM_SHA or run_cmd(["git", "-C", repo, "status", "--porcelain"], capture=True).strip():\n            raise RuntimeError("Upstream checkout changed; use a fresh --root instead of overwriting edits.")\n        run_cmd([python, Path(__file__).resolve(), "_download", "--root", root,\n                 "--model-version", model_version], env=env)\n        frozen = run_cmd([python, "-m", "pip", "freeze"], env=env, capture=True)\n        (root / "environment.freeze.txt").write_text(frozen, encoding="utf-8")\n        print("LATENTSYNC_SETUP_OK", flush=True)\n    return root\n\n\ndef download_models(root, version):\n    # Check the real isolated torch/CUDA combination before downloading ~5 GB\n    # of weights. A mismatched GPU is an error, never an implicit downgrade.\n    gpu_info(version, "auto")\n    from huggingface_hub import hf_hub_download, snapshot_download\n    model = MODELS[version]\n    folder = root / "models" / version\n    for name in ("latentsync_unet.pt", "whisper/tiny.pt"):\n        hf_hub_download(model["repo"], name, revision=model["revision"], local_dir=str(folder))\n    snapshot_download(VAE_REPO, revision=VAE_REVISION, local_dir=str(root / "vae"),\n                      allow_patterns=["config.json", "diffusion_pytorch_model.safetensors"])\n    face_dir = folder / "auxiliary" / "models" / "buffalo_l"\n    if not all((face_dir / name).is_file() and (face_dir / name).stat().st_size > 1024 * 1024 for name in FACE_FILES):\n        archive = root / "buffalo_l.zip"\n        if not archive.exists():\n            temporary = archive.with_suffix(".part")\n            urllib.request.urlretrieve(BUFFALO_URL, temporary)\n            temporary.replace(archive)\n        face_dir.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(archive) as handle:\n            # Extract only known basenames, never trust archive traversal paths.\n            for name in FACE_FILES:\n                member = next((m for m in handle.namelist() if Path(m).name == name), None)\n                if member is None:\n                    raise RuntimeError(f"Official face archive is missing {name}.")\n                with handle.open(member) as source, (face_dir / name).open("wb") as dest:\n                    shutil.copyfileobj(source, dest)\n    write_json(folder / "provenance.json", {"upstream": UPSTREAM_SHA, "model": model,\n               "vae": {"repo": VAE_REPO, "revision": VAE_REVISION},\n               "face_source": BUFFALO_URL,\n               "face_sha256": {name: sha256(face_dir / name) for name in FACE_FILES}})\n\n\ndef gpu_info(version, precision):\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError("No CUDA GPU. In Colab choose Runtime > Change runtime type > GPU.")\n    free, total = torch.cuda.mem_get_info()\n    minimum = MODELS[version]["minimum_gib"]\n    if free / 1024 ** 3 < minimum:\n        raise RuntimeError(f"LatentSync {version} needs at least {minimum} GiB free VRAM; found {free / 1024 ** 3:.1f}. Select 1.5 for T4 or a larger GPU for 1.6.")\n    capability = torch.cuda.get_device_capability()\n    dtype = precision if precision != "auto" else ("fp16" if capability[0] >= 7 else "fp32")\n    return {"name": torch.cuda.get_device_name(), "free_gib": free / 1024 ** 3,\n            "total_gib": total / 1024 ** 3, "capability": list(capability), "precision": dtype,\n            "torch": torch.__version__, "cuda": torch.version.cuda}\n\n\ndef make_stage(directory, root, version):\n    directory.mkdir(parents=True, exist_ok=True)\n    for name, target in {"configs": root / "upstream" / "configs",\n                         "latentsync": root / "upstream" / "latentsync",\n                         "checkpoints": root / "models" / version}.items():\n        (directory / name).symlink_to(target, target_is_directory=True)\n\n\ndef infer_segments(root, workspace, prepared, options, gpu):\n    # Import torch first: ONNX Runtime then finds CUDA/cuDNN shipped with torch.\n    import torch\n    import onnxruntime as ort\n    if hasattr(ort, "preload_dlls"):\n        ort.preload_dlls()\n    from omegaconf import OmegaConf\n    from diffusers import AutoencoderKL, DDIMScheduler\n    from accelerate.utils import set_seed\n    sys.path.insert(0, str(root / "upstream"))\n    from latentsync.models.unet import UNet3DConditionModel\n    from latentsync.pipelines.lipsync_pipeline import LipsyncPipeline\n    from latentsync.whisper.audio2feature import Audio2Feature\n    version = options["model_version"]\n    model = MODELS[version]\n    config = OmegaConf.load(root / "upstream" / model["config"])\n    dtype = torch.float16 if gpu["precision"] == "fp16" else torch.float32\n    audio_encoder = Audio2Feature(model_path=str(root / "models" / version / "whisper" / "tiny.pt"),\n                                 device="cuda", num_frames=16, audio_feat_length=config.data.audio_feat_length)\n    vae = AutoencoderKL.from_pretrained(str(root / "vae"), torch_dtype=dtype, local_files_only=True)\n    vae.config.scaling_factor, vae.config.shift_factor = 0.18215, 0\n    unet, _ = UNet3DConditionModel.from_pretrained(OmegaConf.to_container(config.model),\n                str(root / "models" / version / "latentsync_unet.pt"), device="cpu")\n    pipeline = LipsyncPipeline(vae=vae, audio_encoder=audio_encoder, unet=unet.to(dtype=dtype),\n                scheduler=DDIMScheduler.from_pretrained(str(root / "upstream" / "configs"))).to("cuda")\n    if options["vae_slicing"]:\n        pipeline.enable_vae_slicing()\n    helper = None\n    if options["enable_deepcache"]:\n        from DeepCache import DeepCacheSDHelper\n        helper = DeepCacheSDHelper(pipe=pipeline)\n        helper.set_params(cache_interval=3, cache_branch_id=0)\n        helper.enable()\n    # Generate and report the concrete random seed even for seed=-1.\n    seed = options["seed"] if options["seed"] >= 0 else int.from_bytes(os.urandom(4), "big")\n    outputs = []\n    previous_cwd = Path.cwd()\n    try:\n        for index, (start, duration) in enumerate(prepared["segments"]):\n            stage = workspace / f"segment_{index:05d}"\n            make_stage(stage, root, version)\n            run_cmd(["ffmpeg", "-v", "error", "-nostdin", "-y", "-ss", start,\n                     "-i", prepared["video"], "-t", duration, "-an", "-c:v", "libx264",\n                     "-crf", "13", "-preset", "fast", stage / "input.mp4"])\n            run_cmd(["ffmpeg", "-v", "error", "-nostdin", "-y", "-ss", start,\n                     "-i", prepared["audio"], "-t", duration, "-c:a", "pcm_s16le", stage / "audio.wav"])\n            os.chdir(stage)\n            set_seed((seed + index) % (2 ** 32))\n            pipeline(video_path="input.mp4", audio_path="audio.wav", video_out_path="generated.mp4",\n                     num_frames=16, num_inference_steps=options["inference_steps"],\n                     guidance_scale=options["guidance_scale"], weight_dtype=dtype,\n                     width=model["resolution"], height=model["resolution"],\n                     mask_image_path="latentsync/utils/mask.png", temp_dir="temp")\n            # Upstream does not check its final FFmpeg exit status. This check is\n            # mandatory. Audio features may add ~2 frames; trim before concat.\n            validate_output(stage / "generated.mp4")\n            run_cmd(["ffmpeg", "-v", "error", "-nostdin", "-y", "-i", "generated.mp4",\n                     "-t", duration, "-an", "-c:v", "libx264", "-crf", "13", "-preset", "fast", "trimmed.mp4"])\n            outputs.append(stage / "trimmed.mp4")\n            torch.cuda.empty_cache()\n    finally:\n        os.chdir(previous_cwd)\n        if helper is not None:\n            helper.disable()\n    return outputs, seed\n\n\ndef finalize_media(outputs, prepared, workspace, options):\n    concat = workspace / "concat.txt"\n    # Paths are generated by us, contain no quote/backslash, and are under /content.\n    concat.write_text("".join(f"file \'{path.as_posix()}\'\\n" for path in outputs), encoding="utf-8")\n    final = workspace / "final.mp4"\n    run_cmd(["ffmpeg", "-v", "error", "-nostdin", "-y", "-f", "concat", "-safe", "0",\n             "-i", concat, "-i", prepared["audio"], "-map", "0:v:0", "-map", "1:a:0",\n             "-t", prepared["duration"], "-r", "25", "-c:v", "libx264", "-crf", options["crf"],\n             "-preset", options["preset"], "-pix_fmt", "yuv420p", "-c:a", "aac",\n             "-b:a", options["audio_bitrate"], "-movflags", "+faststart", final])\n    return final\n\n\ndef run_job(root, job_path):\n    root = managed_root(root)\n    job = json.loads(Path(job_path).read_text(encoding="utf-8"))\n    options = resolve_options(job.get("options", {}))\n    video, audio, output = (Path(job[key]).expanduser().resolve() for key in ("video", "audio", "output"))\n    if not video.is_file() or not audio.is_file():\n        raise FileNotFoundError("Video/audio input does not exist in this Colab runtime.")\n    if output in (video, audio) or output.exists():\n        raise ValueError("Output already exists or points at an input; choose a new filename.")\n    with root_lock(root):\n        gpu = gpu_info(options["model_version"], options["precision"])\n        workspace = root / "work" / ("job_" + uuid.uuid4().hex)\n        workspace.mkdir(parents=True)\n        started = time.time()\n        report = {"status": "running", "job_id": job.get("id"), "options": options,\n                  "gpu": gpu, "upstream_sha": UPSTREAM_SHA, "model": MODELS[options["model_version"]],\n                  "workspace": str(workspace), "input_video": str(video), "input_audio": str(audio)}\n        report_path = Path(str(output) + ".report.json")\n        try:\n            prepared = prepare_media(video, audio, workspace, options)\n            outputs, seed = infer_segments(root, workspace, prepared, options, gpu)\n            final = finalize_media(outputs, prepared, workspace, options)\n            metadata = validate_output(final, prepared["duration"])\n            output.parent.mkdir(parents=True, exist_ok=True)\n            # Exclusive creation avoids overwriting a file that appeared mid-run.\n            with final.open("rb") as source, output.open("xb") as dest:\n                shutil.copyfileobj(source, dest)\n            report.update(status="completed", effective_seed=seed,\n                          duration=prepared["duration"], segments=prepared["segments"],\n                          output=str(output), output_sha256=sha256(output), output_metadata=metadata,\n                          elapsed_seconds=round(time.time() - started, 3),\n                          provenance=json.loads((root / "models" / options["model_version"] / "provenance.json").read_text()))\n            write_json(report_path, report)\n            print("LATENTSYNC_JOB_OK:" + str(job.get("id", "completed")), flush=True)\n        except BaseException as exc:\n            report.update(status="failed", error=f"{type(exc).__name__}: {exc}",\n                          elapsed_seconds=round(time.time() - started, 3))\n            write_json(report_path, report)\n            print(f"Job failed; intermediates retained at {workspace}", file=sys.stderr, flush=True)\n            raise\n        else:\n            if not options["keep_intermediates"]:\n                # Only delete the uniquely generated child, never arbitrary input.\n                if workspace.parent == root / "work" and workspace.name.startswith("job_"):\n                    shutil.rmtree(workspace)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest="command", required=True)\n    for name in ("setup", "run", "_download", "_worker"):\n        command = commands.add_parser(name)\n        command.add_argument("--root", default="/content/latentsync-skill")\n        if name in ("setup", "_download"):\n            command.add_argument("--model-version", choices=list(MODELS), default="1.6")\n        else:\n            command.add_argument("--job", required=True)\n    args = parser.parse_args(argv)\n    if args.command == "setup":\n        setup(args.root, args.model_version)\n    elif args.command == "_download":\n        download_models(Path(args.root), args.model_version)\n    elif args.command == "_worker":\n        run_job(args.root, args.job)\n    else:\n        job_path = Path(args.job).resolve()\n        job = json.loads(job_path.read_text(encoding="utf-8"))\n        options = resolve_options(job.get("options", {}))\n        root = setup(args.root, options["model_version"])\n        run_cmd([runtime_python(root), Path(__file__).resolve(), "_worker", "--root", root,\n                 "--job", job_path], env=child_env(root))\n\n\nif __name__ == "__main__":\n    try:\n        main()\n    except (Exception, KeyboardInterrupt) as error:\n        traceback.print_exc()\n        print(f"ERROR: {type(error).__name__}: {error}", file=sys.stderr, flush=True)\n        raise SystemExit(1)\n'}
for name, source in SOURCES.items():
    (SCRIPT_DIR / name).write_text(source, encoding='utf-8')
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
from config import resolve_options
RUNTIME_ROOT = '/content/latentsync-skill'

def run_remote(arguments):
    # Relay child output through Python stdout so Colab displays it in the cell.
    with subprocess.Popen(arguments, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                          errors='replace', bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

print('工具已準備。')


In [ ]:
#@title 2. 模型與媒體參數（先用預設值）
model_version = "1.6" #@param ["1.6", "1.5"]
inference_steps = 20 #@param {type:"integer"}
guidance_scale = 1.5 #@param {type:"number"}
seed = 1247 #@param {type:"integer"}
precision = "auto" #@param ["auto", "fp16", "fp32"]
enable_deepcache = False #@param {type:"boolean"}
vae_slicing = False #@param {type:"boolean"}
video_start = 0.0 #@param {type:"number"}
audio_start = 0.0 #@param {type:"number"}
duration_seconds = 5.0 #@param {type:"number"}
# duration_seconds=0 表示不另設上限；首次建議5秒。
length_mode = "shortest" #@param ["shortest", "audio", "video"]
audio_offset = 0.0 #@param {type:"number"}
audio_gain_db = 0.0 #@param {type:"number"}
max_side = 1280 #@param {type:"integer"}
segment_seconds = 0.0 #@param {type:"number"}
crf = 18 #@param {type:"integer"}
preset = "medium" #@param ["ultrafast", "superfast", "veryfast", "faster", "fast", "medium", "slow", "slower", "veryslow"]
audio_bitrate = "192k" #@param ["96k", "128k", "160k", "192k", "256k", "320k"]
keep_intermediates = False #@param {type:"boolean"}
options = resolve_options({
    'model_version': model_version, 'inference_steps': inference_steps,
    'guidance_scale': guidance_scale, 'seed': seed, 'precision': precision,
    'enable_deepcache': enable_deepcache, 'vae_slicing': vae_slicing,
    'video_start': video_start, 'audio_start': audio_start,
    'duration': None if duration_seconds == 0 else duration_seconds,
    'length_mode': length_mode, 'audio_offset': audio_offset,
    'audio_gain_db': audio_gain_db, 'max_side': max_side,
    'segment_seconds': segment_seconds, 'crf': crf, 'preset': preset,
    'audio_bitrate': audio_bitrate, 'keep_intermediates': keep_intermediates,
})
print(json.dumps(options, ensure_ascii=False, indent=2))


In [ ]:
#@title 3. 安裝環境與下載模型（首次較久；重複執行會檢查快取）
run_remote([sys.executable, str(SCRIPT_DIR / 'remote_runtime.py'),
            'setup', '--root', RUNTIME_ROOT,
            '--model-version', options['model_version']])


In [ ]:
#@title 4. 選擇輸入方式並上傳
input_mode = "upload" #@param ["upload", "paths"]
video_path = "/content/video.mp4" #@param {type:"string"}
audio_path = "/content/audio.wav" #@param {type:"string"}
# paths可使用已上傳檔案，或自行mount後的Google Drive路徑。
from google.colab import files
if input_mode == 'upload':
    print('先選一個影片檔')
    video_upload = files.upload()
    if len(video_upload) != 1:
        raise ValueError('請只選一個影片檔')
    video_path = str(Path(next(iter(video_upload))).resolve())
    del video_upload
    print('再選一個替換音檔（檔名不能與影片相同）')
    audio_upload = files.upload()
    if len(audio_upload) != 1:
        raise ValueError('請只選一個音檔')
    audio_path = str(Path(next(iter(audio_upload))).resolve())
    del audio_upload
if Path(video_path).resolve() == Path(audio_path).resolve():
    raise ValueError('影片與音檔必須是不同路徑')
for path in (video_path, audio_path):
    if not Path(path).is_file() or Path(path).stat().st_size == 0:
        raise ValueError('檔案不存在或空白: ' + path)
print('影片:', video_path, '\n音檔:', audio_path)


In [ ]:
#@title 5. 執行嘴型同步
OUTPUT = Path('/content/outputs') / ('latentsync16_' + uuid.uuid4().hex[:8] + '.mp4')
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
job_file = SCRIPT_DIR / 'job.json'
job_file.write_text(json.dumps({'video': video_path, 'audio': audio_path,
                               'output': str(OUTPUT), 'options': options}), encoding='utf-8')
run_remote([sys.executable, str(SCRIPT_DIR / 'remote_runtime.py'),
            'run', '--root', RUNTIME_ROOT, '--job', str(job_file)])
if not OUTPUT.is_file() or OUTPUT.stat().st_size == 0:
    raise RuntimeError('未產生成品，請看上方錯誤')
print('完成:', OUTPUT)


In [ ]:
#@title 6. 預覽與下載（完成後記得刪除執行階段）
from IPython.display import Video, display
display(Video(str(OUTPUT), embed=True))
files.download(str(OUTPUT))
report = Path(str(OUTPUT) + '.report.json')
if report.is_file():
    files.download(str(report))
print('請確認MP4已下載，再到「執行階段 → 中斷連線並刪除執行階段」。')
